# Agent 3: bardziej złożony agent wieloetapowy

To jest **bardziej rozbudowany agent zewnętrzny**, nadal niepłatny i możliwy do uruchomienia lokalnie.

## Cechy
- planowanie etapów,
- delegowanie do subagentów,
- mini-RAG na lokalnych dokumentach,
- pamięć robocza,
- raport końcowy z uzasadnieniem.

Architektura:
1. **PlannerAgent** tworzy plan.
2. **ResearchAgent** wyszukuje wiedzę w lokalnych dokumentach.
3. **WriterAgent** buduje końcową odpowiedź.
4. **Orchestrator** spina wszystko razem.

To już przypomina mały system typu:
**manager agent + workers**

In [ ]:
!apt-get update -y
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!nohup ollama serve > /tmp/ollama.log 2>&1 &

In [ ]:
import time
time.sleep(8)

In [ ]:
USE_OLLAMA = True   # zmień na True, jeśli masz uruchomione Ollama lokalnie
OLLAMA_MODEL = "gemma2:2b"

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Any
import re
from collections import Counter
import textwrap

In [ ]:
class MockLLM:
    def generate(self, prompt: str) -> str:
        p = prompt.lower()
        if "utwórz plan" in p:
            return "1. Zidentyfikuj temat\n2. Wyszukaj lokalne źródła\n3. Zsyntetyzuj odpowiedź\n4. Podaj rekomendacje"
        if "napisz raport końcowy" in p:
            return (
                "Raport końcowy:\n"
                "- Temat został przeanalizowany etapowo.\n"
                "- Wykorzystano lokalne źródła wiedzy.\n"
                "- Na końcu wygenerowano praktyczne rekomendacje."
            )
        return "Odpowiedź wygenerowana przez MockLLM."

class OllamaLLM:
    def __init__(self, model: str, url: str = "http://localhost:11434/api/generate"):
        self.model = model
        self.url = url
    def generate(self, prompt: str) -> str:
        import requests
        payload = {"model": self.model, "prompt": prompt, "stream": False}
        resp = requests.post(self.url, json=payload, timeout=120)
        resp.raise_for_status()
        return resp.json()["response"]

def get_llm(use_ollama: bool, model: str):
    return OllamaLLM(model) if use_ollama else MockLLM()

In [ ]:
LOCAL_DOCS = [
    {"id": "doc1", "text": "RAG łączy wyszukiwanie informacji z generowaniem odpowiedzi na podstawie kontekstu."},
    {"id": "doc2", "text": "Agent może korzystać z narzędzi, pamięci, planowania i subagentów do realizacji celu."},
    {"id": "doc3", "text": "W dydaktyce NLP warto pokazać różnicę między chatbotem, agentem i systemem wieloagentowym."},
    {"id": "doc4", "text": "Modele lokalne można uruchamiać przez Ollama, co pozwala budować niepłatne demonstracje agentów."}
]

def tokenize(text: str):
    return re.findall(r"\w+", text.lower())

def retrieve_docs(query: str, top_k: int = 3):
    q = Counter(tokenize(query))
    scored = []
    for doc in LOCAL_DOCS:
        d = Counter(tokenize(doc["text"]))
        overlap = sum((q & d).values())
        scored.append((overlap, doc))
    scored.sort(reverse=True, key=lambda x: x[0])
    return [doc for score, doc in scored[:top_k] if score > 0]

In [ ]:
@dataclass
class PlannerAgent:
    llm: Any
    def plan(self, goal: str) -> str:
        prompt = f"Utwórz plan realizacji zadania. Cel: {goal}"
        return self.llm.generate(prompt)

@dataclass
class ResearchAgent:
    def research(self, goal: str) -> List[Dict[str, str]]:
        return retrieve_docs(goal, top_k=3)

@dataclass
class WriterAgent:
    llm: Any
    def write(self, goal: str, plan: str, docs: List[Dict[str, str]]) -> str:
        docs_text = "\n".join([f"- {d['id']}: {d['text']}" for d in docs])
        prompt = textwrap.dedent(f'''
        Napisz raport końcowy po polsku.

        Cel:
        {goal}

        Plan:
        {plan}

        Źródła:
        {docs_text}

        Raport ma być krótki, konkretny i praktyczny.
        ''')
        return self.llm.generate(prompt)

In [ ]:
@dataclass
class OrchestratorAgent:
    planner: PlannerAgent
    researcher: ResearchAgent
    writer: WriterAgent
    working_memory: Dict[str, Any] = field(default_factory=dict)

    def run(self, goal: str) -> Dict[str, Any]:
        plan = self.planner.plan(goal)
        docs = self.researcher.research(goal)
        report = self.writer.write(goal, plan, docs)

        self.working_memory["goal"] = goal
        self.working_memory["plan"] = plan
        self.working_memory["docs"] = docs
        self.working_memory["report"] = report

        return self.working_memory

In [ ]:
llm = get_llm(USE_OLLAMA, OLLAMA_MODEL)

system = OrchestratorAgent(
    planner=PlannerAgent(llm),
    researcher=ResearchAgent(),
    writer=WriterAgent(llm),
)

result = system.run("Przygotuj krótkie wyjaśnienie różnicy między chatbotem, agentem i systemem wieloagentowym oraz wskaż rolę RAG.")

print("PLAN:")
print(result["plan"])
print("\nDOKUMENTY:")
for d in result["docs"]:
    print(d)
print("\nRAPORT:")
print(result["report"])

assert "goal" in result and "report" in result
assert isinstance(result["docs"], list)
assert len(result["docs"]) >= 1
assert isinstance(result["report"], str) and len(result["report"]) > 20

PLAN:
1. Zidentyfikuj temat
2. Wyszukaj lokalne źródła
3. Zsyntetyzuj odpowiedź
4. Podaj rekomendacje

DOKUMENTY:
{'id': 'doc3', 'text': 'W dydaktyce NLP warto pokazać różnicę między chatbotem, agentem i systemem wieloagentowym.'}
{'id': 'doc1', 'text': 'RAG łączy wyszukiwanie informacji z generowaniem odpowiedzi na podstawie kontekstu.'}
{'id': 'doc2', 'text': 'Agent może korzystać z narzędzi, pamięci, planowania i subagentów do realizacji celu.'}

RAPORT:
Raport końcowy:
- Temat został przeanalizowany etapowo.
- Wykorzystano lokalne źródła wiedzy.
- Na końcu wygenerowano praktyczne rekomendacje.


## Rozszerzenia na dalsze zajęcia
- dodać prawdziwe embeddings,
- użyć FAISS / ChromaDB,
- przejść do `LangGraph` albo własnej maszyny stanów,
- dodać narzędzia typu pliki, web, SQL, kalendarz.